In [5]:
%load_ext autoreload
%autoreload 2

import os
import yaml
from balance_metrics import get_data, compute_correlations

data_path = "../../experiment_data/balance_metrics/tvcg/entropy_1e6.csv"

data = get_data(data_path)
data = data[data["layer"] == "a1"]  # Filter to only penultimate layer for correlations
compute_correlations(data, data_path.replace(".csv", "_penultimate_corr.csv"))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Correlation results saved to ../../experiment_data/balance_metrics/tvcg/entropy_1e6_penultimate_corr.csv


In [6]:
# epoch_trainUval = pd.concat([epoch_data[(epoch_data['split'] == 'trainUval')], epoch_data_better[(epoch_data_better['split'] == 'trainUval')]])
epoch_trainUval = pd.concat([data[(data['split'] == 'trainUval')]])
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig = make_subplots(specs=[[{"secondary_y": True}]], )

sackin_scatter_trainUval = px.scatter(epoch_trainUval, x='epoch', y='sackin_index', color="dataset", color_discrete_sequence=color_seq, symbol="model")
train_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", color_discrete_sequence=color_seq_train, symbol="model")
val_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", color_discrete_sequence=color_seq_val, symbol="model", line_dash_sequence=['dash'])

for i in range(len(tuple(sackin_scatter_trainUval.data))):
	# get dataset, model out of the trace
	dataset, model = str(sackin_scatter_trainUval.data[i]['name']).split(", ")
    
	all_epochs = epoch_trainUval[(epoch_trainUval['dataset'] == dataset) & (epoch_trainUval['model'] == model)]
	# find the epoch where val acc is maximum
	best_epoch = all_epochs.sort_values(by='val_acc', ascending=False).iloc[0]['epoch']

	fig.add_trace(
		sackin_scatter_trainUval.data[i],
		secondary_y=False,
	)

	# fig.update_yaxes(type="log", secondary_y=False)

	fig.add_trace(
		train_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	fig.add_trace(
		val_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	# add a vertical line at best epoch
	fig.add_trace(go.Scatter(x=[best_epoch, best_epoch], y=[0, 1], mode="lines", line=dict(color=sackin_scatter_trainUval.data[i]['marker']['color'], dash="dot"), legendgroup=sackin_scatter_trainUval.data[i]['name'], showlegend=False), secondary_y=True)

# set title
fig.update_layout(
	title_text="Cross Epoch on TrainUVal: Sackin Index vs Train/Val Accuracy"
)
fig.show()